# Projeto 01 - Agente de Rotas para Comida Local (IA 2026-27)

*Inteligência Artificial - Licenciatura em Engenharia de Sistemas Informáticos (UPCA / EST)*

### Contexto
Uma plataforma digital liga **produtores** locais, **clientes** e **centros de recolha (armazéns)**.
Cada cliente faz uma encomenda e escolhe um **local de entrega**. Os produtos são recolhidos nos produtores,
passam obrigatoriamente pelo armazém (`producer > warehouse > delivery_location`) e são depois entregues,
respeitando:
* a **janela de disponibilidade** do produto no produtor (`availability_start / availability_end`);
* a **janela de entrega** no local de destino (`delivery_start / delivery_end`);
* a **capacidade** dos veículos.

O objetivo do projeto é desenvolver um agente que escolha as melhores rotas de recolha e entrega usando
**pesquisa informada** (como Greedy, A*), **pesquisa local** (como Hill-Climbing) e **CSP** (biblioteca `python-constraint`).

### Equipa (Grupo G04)
| Nome | Número |
|------|--------|
| Diogo Apolinário | 6731 |
| Raúl Ribeiro | 22552 |
| Miguel Correia | 31028 |
| José Ferreira | 31556 |

**Repositório GitHub:** https://github.com/JoseFerreira31556/IA26_P01_G04.git

### Este notebook - Entrega 1: Leitura dos ficheiros e contagem de linhas
O pandas lê os ficheiros CSV do dataset, conta as linhas de cada um e imprime esse valor.

In [ ]:
# Importar bibliotecas
import pandas
import os

In [ ]:
path = "./data/data_tiny_20261020/"  # path to the folder containing the CSV files
files_list = sorted(os.listdir(path)) # list of CSV files in the folder

def read_csv_files(path, files_list):    
    data = {} #dicionário com a informação dos ficheiros

    for file in files_list:
        if not file.endswith(".csv"): continue

        # Read the CSV file into a pandas dataframe
        rf = pandas.read_csv(os.path.join(path,file))

        name = file.replace(".csv", "")   
        data[name] = rf
        
        #rows -> line number
        #columns -> column number

        # rows, columns = rf.shape

        # print(file + ": " + str(rows) + " rows")  

    return data

data = read_csv_files(path, files_list)

### Verificação de incongruências nos dados
Pretende-se com este bloco verificar se existem valores em falta, linhas duplicadas e se o tipo corresponde à informação observada na coluna. 
Foram usados os seguintes métodos do pandas:
*  **items()**: permite tirar do dicionário a key e o conteúdo;
*  **isna()**: verifica se existe alguma célula vazia, atribuindo um True em caso verdadeiro e um False caso não esteja
*  **duplicated()**: verificação da existência de duplicados
*  **sum()**: soma dos valores que deu na tabela. Considera o True como 1 e o False como 0
*  **duplicated(subset=[])**: permite escolher as colunas que se querem comparar

In [ ]:
#data.items() gives the name of the file and it's inside information
for name, table in data.items():         
    #Variable to check for empty cells
    sumEmptyCells = table.isna().sum().sum()

    #Variable to check for duplicated cells
    duplicates = table.duplicated().sum()   

    #Verifying if there is empty cells
    if(sumEmptyCells != 0): print("Missing a value in file " + name)

    #Verifying if there is duplicated cells
    if(duplicates != 0): print("File " + name + " with duplicated information") 

    #Variable that checks only the first column of the table
    tableIdColumn = table.columns[0]  

    isId = tableIdColumn.endswith("_id")
    if(isId == False): continue

    if(name == "distance_matrix"):
        duplicatesDM = table.duplicated(subset=["origin_location_id", "destination_location_id"]).sum()
        if(duplicatesDM != 0 ) : 
            print("Duplicated lines in file " + name)            
        continue

    #Variable that counts the number of duplicated id's
    duplicatedIds= table[tableIdColumn].duplicated().sum()

    #Verifying if there is duplicated id'S
    if(duplicatedIds != 0): print("File " + name + " with duplicated id")   
                             


### Conversão de datas em string para datetime
Pretende-se com este bloco converter as datas em datetime. Para isso foi usado um método do pandas: 
*  **pandas.to_datetime()**: converte as variáveis para o tipo *datetime(us)*

In [ ]:
for name, table in data.items(): 
    #Converting dates (strings to datetime)
    for column in table.columns:
        if(column.endswith(("_end", "_start", "_datetime"))):
            dateColumns = table[column]
            table[column] = pandas.to_datetime(dateColumns)          
    